In [5]:
%pip install qiskit qiskit-aer qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [qiskit-ibm-runtime]iskit-ibm-runtime]es]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Python `random.randint` (Pseudorandom)

In [6]:
import random

print("=== Python random.randint (Pseudorandom - Mersenne Twister) ===\n")
for pair in range(5):
    nums = [random.randint(1, 10) for _ in range(10)]
    print(f"Pair {pair + 1}: {nums}")

=== Python random.randint (Pseudorandom - Mersenne Twister) ===

Pair 1: [1, 2, 7, 5, 1, 10, 1, 8, 10, 8]
Pair 2: [2, 10, 6, 7, 8, 10, 8, 6, 8, 8]
Pair 3: [6, 6, 3, 10, 8, 7, 7, 9, 9, 3]
Pair 4: [2, 8, 6, 3, 4, 9, 7, 3, 1, 10]
Pair 5: [3, 8, 7, 5, 4, 4, 1, 2, 4, 6]


## 2. Qiskit Local Simulator (Simulated Quantum)

In [7]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator


def quantum_random_numbers(low, high, count):
    """Generate random integers using quantum superposition."""
    n_bits = (high - low).bit_length()
    results = []

    simulator = AerSimulator()

    while len(results) < count:
        qc = QuantumCircuit(n_bits, n_bits)
        qc.h(range(n_bits))
        qc.measure(range(n_bits), range(n_bits))

        job = simulator.run(qc, shots=count * 3)
        counts = job.result().get_counts()

        for bitstring in counts:
            val = int(bitstring, 2) + low
            if low <= val <= high:
                results.append(val)
                if len(results) >= count:
                    break

    return results[:count]


print("=== Qiskit Local Simulator (Simulated Quantum) ===\n")
for pair in range(5):
    nums = quantum_random_numbers(1, 10, 10)
    print(f"Pair {pair + 1}: {nums}")

Pair 1: [10, 7, 5, 9, 1, 3, 8, 4, 8, 6]
Pair 2: [7, 9, 1, 3, 10, 8, 2, 5, 6, 1]
Pair 3: [1, 7, 9, 10, 3, 8, 4, 2, 6, 10]
Pair 4: [10, 5, 9, 2, 4, 1, 7, 10, 7, 6]
Pair 5: [8, 3, 1, 2, 4, 9, 5, 7, 6, 10]


## 3. Real IBM Quantum Hardware (True Quantum Randomness)

Sign up for a free account at [quantum.ibm.com](https://quantum.ibm.com/) and copy your API token from the dashboard. Paste it below.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

IBM_QUANTUM_TOKEN = "PASTE_YOUR_TOKEN_HERE"

QiskitRuntimeService.save_account(channel="ibm_quantum", token=IBM_QUANTUM_TOKEN, overwrite=True)
service = QiskitRuntimeService(channel="ibm_quantum")
backend = service.least_busy(simulator=False, operational=True)
print(f"Using real quantum backend: {backend.name}")


def real_quantum_random_numbers(low, high, count, backend):
    """Generate random integers on real IBM quantum hardware."""
    n_bits = (high - low).bit_length()
    results = []

    qc = QuantumCircuit(n_bits, n_bits)
    qc.h(range(n_bits))
    qc.measure(range(n_bits), range(n_bits))

    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    transpiled = pm.run(qc)

    sampler = Sampler(mode=backend)
    job = sampler.run([transpiled], shots=count * 5)
    result = job.result()
    counts = result[0].data.meas.get_counts()

    for bitstring, freq in counts.items():
        val = int(bitstring, 2) + low
        if low <= val <= high:
            results.extend([val] * freq)
            if len(results) >= count:
                break

    return results[:count]


print("\n=== Real IBM Quantum Hardware (True Quantum Randomness) ===\n")
for pair in range(5):
    nums = real_quantum_random_numbers(1, 10, 10, backend)
    print(f"Pair {pair + 1}: {nums}")